#### Postal codes dataset

https://www.data.gouv.fr/api/1/datasets/r/c63fd0b1-7987-46f6-b779-8b3ed889090c

#### Rent dataset

https://www.data.gouv.fr/api/1/datasets/r/55b34088-0964-415f-9df7-d87dd98a09be

#### Income dataset

https://www.data.gouv.fr/api/1/datasets/r/6d7542dd-4aa7-4683-928e-df4292f5cd53

#### Libraries dataset

https://www.data.gouv.fr/api/1/datasets/r/806a8aa1-952f-404d-9857-3f27b7c0ca86

#### Bookshop dataset

https://www.data.gouv.fr/api/1/datasets/r/bcd7d584-89c3-46b6-a164-4582eb2d43b5

In [ ]:
import re
import struct
import zipfile
from io import BytesIO

import pandas as pd
import requests

SOURCES = {
    "postal_codes": "https://www.data.gouv.fr/api/1/datasets/r/c63fd0b1-7987-46f6-b779-8b3ed889090c",
    "rent": "https://www.data.gouv.fr/api/1/datasets/r/55b34088-0964-415f-9df7-d87dd98a09be",
    "income": "https://www.data.gouv.fr/api/1/datasets/r/6d7542dd-4aa7-4683-928e-df4292f5cd53",
    "libraries": "https://www.data.gouv.fr/api/1/datasets/r/806a8aa1-952f-404d-9857-3f27b7c0ca86",
    "bookshops": "https://www.data.gouv.fr/api/1/datasets/r/bcd7d584-89c3-46b6-a164-4582eb2d43b5",
}


def read_dbf(raw):
    """Read a DBF attribute table stored inside a shapefile ZIP archive."""
    record_count = struct.unpack("<I", raw[4:8])[0]
    header_length = struct.unpack("<H", raw[8:10])[0]
    record_length = struct.unpack("<H", raw[10:12])[0]

    fields = []
    position = 32

    while raw[position] != 13:
        field = raw[position : position + 32]
        name = field[:11].split(b"\0")[0].decode("latin1")
        fields.append((name, chr(field[11]), field[16], field[17]))
        position += 32

    rows = []

    for row_number in range(record_count):
        start = header_length + row_number * record_length
        record = raw[start : start + record_length]

        if not record or record[0] == 42:  # Deleted record
            continue

        values = {}
        offset = 1

        for field_name, field_type, width, decimals in fields:
            text = record[offset : offset + width].decode(
                "cp1252", errors="replace"
            ).strip()
            offset += width

            if not text or set(text) == {"*"}:
                values[field_name] = pd.NA
            elif field_type in "NF":
                values[field_name] = float(text) if decimals else int(text)
            else:
                values[field_name] = text

        rows.append(values)

    return pd.DataFrame(rows)


def download_table(name, url):
    response = requests.get(url, timeout=120)
    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "").lower()
    data = BytesIO(response.content)

    # Income source is a ZIP containing a shapefile/DBF table.
    if zipfile.is_zipfile(data):
        data.seek(0)

        with zipfile.ZipFile(data) as archive:
            dbf_files = [
                file for file in archive.namelist()
                if file.lower().endswith(".dbf")
            ]

            if dbf_files:
                return read_dbf(archive.read(dbf_files[0]))

        data.seek(0)

    if "excel" in content_type or response.content[:2] == b"PK":
        return pd.read_excel(data)

    if "json" in content_type:
        return pd.read_json(data)

    for encoding in ("utf-8", "cp1252", "latin1"):
        try:
            data.seek(0)
            return pd.read_csv(
                data,
                sep=None,
                engine="python",
                encoding=encoding,
            )
        except UnicodeDecodeError:
            continue

    raise ValueError(f"Could not decode {name} as a CSV file.")


def normalise_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def postal_column(df):
    candidates = {
        "postalcode",
        "postcode",
        "zipcode",
        "zip",
        "codepostal",
    }

    return next(
        (column for column in df.columns if normalise_name(column) in candidates),
        None,
    )


def clean_postalcode(values):
    return (
        values.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
        .where(values.notna())
    )


# Download all datasets
tables = {}

for name, url in SOURCES.items():
    tables[name] = download_table(name, url)
    print(f"Downloaded {name}: {tables[name].shape[0]:,} rows")


# First dataset is the reference dataset
big_dataset = tables["postal_codes"].copy()

base_key = postal_column(big_dataset)

if base_key is None:
    raise ValueError("The postal-code dataset has no postal-code column.")

big_dataset = big_dataset.rename(columns={base_key: "postalcode"})
big_dataset["postalcode"] = clean_postalcode(big_dataset["postalcode"])


# Merge datasets only when they contain a postal-code column
for name, table in tables.items():
    if name == "postal_codes":
        continue

    table = table.copy()
    key = postal_column(table)

    if key is None:
        print(f"Skipped {name}: no postal-code column.")
        continue

    table = table.rename(columns={key: "postalcode"})
    table["postalcode"] = clean_postalcode(table["postalcode"])

    # Prevent duplicate column names.
    table = table.rename(
        columns={
            column: f"{name}_{column}"
            for column in table.columns
            if column != "postalcode"
        }
    )

    # One row per postal code before merging.
    table = table.groupby("postalcode", dropna=False).first().reset_index()

    big_dataset = big_dataset.merge(
        table,
        on="postalcode",
        how="left",
    )


big_dataset.to_csv(
    "big_merged_dataset.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: big_merged_dataset.csv")
print(big_dataset.head())

Downloaded postal_codes: 34,868 rows
Downloaded rent: 34,900 rows
Downloaded income: 34,926 rows
Downloaded libraries: 15,704 rows
Downloaded bookshops: 3,237 rows
Skipped rent: no postal-code column.
Skipped income: no postal-code column.
Skipped libraries: no postal-code column.
Saved: big_merged_dataset.csv
  code_insee             nom_standard        nom_sans_pronom  \
0      01001  L'Abergement-Clémenciat  Abergement-Clémenciat   
1      01002    L'Abergement-de-Varey    Abergement-de-Varey   
2      01004        Ambérieu-en-Bugey      Ambérieu-en-Bugey   
3      01005      Ambérieux-en-Dombes    Ambérieux-en-Dombes   
4      01006                  Ambléon                Ambléon   

                       nom_a                      nom_de  \
0  à l'Abergement-Clémenciat  de l'Abergement-Clémenciat   
1    à l'Abergement-de-Varey    de l'Abergement-de-Varey   
2        à Ambérieu-en-Bugey         d'Ambérieu-en-Bugey   
3      à Ambérieux-en-Dombes       d'Ambérieux-en-Dombes   
4  

In [15]:
rows_with_missing = big_dataset.isna().any(axis=1).sum()

rows_without_missing = big_dataset.notna().all(axis=1).sum()

print(f"Rows with at least one missing value: {rows_with_missing:,}")
print(f"Rows with no missing values: {rows_without_missing:,}")

Rows with at least one missing value: 33,935
Rows with no missing values: 933


In [13]:
rows_with_missing_data = big_dataset[big_dataset.isna().any(axis=1)]
rows_with_missing_data.head()

,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,reg_code,...,gentile,url_wikipedia,url_villedereve,bookshops_﻿librairie,bookshops_assortiment,bookshops_label,bookshops_Commune,bookshops_code_commune,bookshops_Departement,bookshops_Region
0,01001,L'Abergement-Clémenciat,Abergement-Clémenciat,à l'Abergement-Clémenciat,de l'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,COM,commune,84,...,NaN,https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01001-l-abergemen...,D EN BULLE,BD et Mangas,None,CHATILLON SUR CHALARONNE,01093,AIN,AUVERGNE RHONE ALPES
1,01002,L'Abergement-de-Varey,Abergement-de-Varey,à l'Abergement-de-Varey,de l'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,COM,commune,84,...,"Abergementais, Abergementaises",https://fr.wikipedia.org/wiki/fr:L'Abergement-...,https://villedereve.fr/ville/01002-l-abergemen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,à Ambérieu-en-Bugey,d'Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,COM,commune,84,...,"Ambarrois, Ambarroises",https://fr.wikipedia.org/wiki/fr:Ambérieu-en-B...,https://villedereve.fr/ville/01004-amberieu-en...,BLANCHE NEIGE,Généraliste,None,AMBERIEU EN BUGEY,01004,AIN,AUVERGNE RHONE ALPES
3,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,à Ambérieux-en-Dombes,d'Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,COM,commune,84,...,Ambarrois,https://fr.wikipedia.org/wiki/fr:Ambérieux-en-...,https://villedereve.fr/ville/01005-amberieux-e...,DOMINO,Généraliste,None,VILLARS LES DOMBES,01443,AIN,AUVERGNE RHONE ALPES
4,01006,Ambléon,Ambléon,à Ambléon,d'Ambléon,ambleon,AMBLÉON,COM,commune,84,...,Ambléonais,https://fr.wikipedia.org/wiki/fr:Ambléon,https://villedereve.fr/ville/01006-ambleon,ENTRE PARENTHESES,Généraliste,None,BELLEY,01034,AIN,AUVERGNE RHONE ALPES
